In [0]:

tables = [
    "bronze_smart_meter_readings",
    "bronze_grid_outage_events",
    "bronze_meter_recovery_events",
    "bronze_meter_master",
    "bronze_substation_master",
    "bronze_tariff_lookup"
]

for table in tables:
    full_name = f"capstone9_smartmeter.smart_grid_bronze.{table}"

    print("\n" + "=" * 70)
    print("TABLE:", table)

    df = spark.table(full_name)

    print("Record count:", df.count())
    df.printSchema()
    display(df.limit(5))


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# 1. CATALOG AND SCHEMA
# --------------------------------------------------

catalog = "capstone9_smartmeter"
bronze_schema = "smart_grid_bronze"
silver_schema = "smart_grid_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

In [0]:
# --------------------------------------------------
# 2. HELPER FUNCTIONS
# --------------------------------------------------

def read_bronze(table):
    df = spark.table(f"{catalog}.{bronze_schema}.{table}")

    # Trim all string columns
    for c, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(c, F.trim(F.col(c)))

    return df


def safe_cast(df, column, data_type):
    # Invalid conversions become NULL instead of failing the whole job
    return df.withColumn(
        column,
        F.expr(f"try_cast(`{column}` AS {data_type})")
    )


def standardize_flag(df, column):
    value = F.lower(F.trim(F.col(column)))

    return df.withColumn(
        column,
        F.when(value.isin("true", "1", "yes", "y", "active"), True)
         .when(value.isin("false", "0", "no", "n", "inactive"), False)
         .otherwise(None)
    )


def deduplicate(df, keys, order_column="_ingested_at"):
    window = Window.partitionBy(*keys).orderBy(
        F.col(order_column).desc_nulls_last()
    )

    return (
        df.withColumn("_row_number", F.row_number().over(window))
          .filter(F.col("_row_number") == 1)
          .drop("_row_number")
    )


def publish(df, silver_table, required_columns, rules=None):
    # rules: list of (condition, reason) tuples. A row is quarantined with
    # the given reason when its condition evaluates to False.
    # Build data quality rejection reasons
    # Empty-string check only applies to string columns (comparing a
    # DATE/TIMESTAMP to "" fails under ANSI mode)
    col_types = dict(df.dtypes)

    def is_missing(c):
        if col_types[c] == "string":
            return F.col(c).isNull() | (F.col(c) == "")
        return F.col(c).isNull()

    reasons = [
        F.when(is_missing(c), F.lit(f"Null validation failed: {c}"))
        for c in required_columns
    ]

    for condition, reason in (rules or []):
        reasons.append(F.when(~condition, F.lit(reason)))

    df = df.withColumn(
        "_dq_reason",
        F.concat_ws(
            "; ",
            F.array_remove(F.array(*reasons), "")
        )
    )

    valid_df = df.filter(F.col("_dq_reason") == "").drop("_dq_reason")
    rejected_df = df.filter(F.col("_dq_reason") != "")

    # Write clean Silver table
    (
        valid_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{silver_schema}.{silver_table}")
    )

    # Write rejected records separately
    quarantine_table = silver_table.replace("silver_", "quarantine_")

    (
        rejected_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{silver_schema}.{quarantine_table}")
    )

    print(
        f"{silver_table}: Valid = {valid_df.count()}, "
        f"Quarantined = {rejected_df.count()}"
    )

    return valid_df

In [0]:
# --------------------------------------------------
# 3. SUBSTATION MASTER
# --------------------------------------------------

substation = read_bronze("bronze_substation_master")

substation = safe_cast(substation, "capacity_mva", "DOUBLE")
substation = safe_cast(substation, "voltage_level_kv", "DOUBLE")
substation = safe_cast(substation, "commissioned_date", "DATE")
substation = safe_cast(substation, "latitude", "DOUBLE")
substation = safe_cast(substation, "longitude", "DOUBLE")
substation = standardize_flag(substation, "active_flag")

substation = deduplicate(substation, ["substation_id"])

silver_substation = publish(
    substation,
    "silver_substation_master",
    ["substation_id", "substation_name"],
    [
        (F.col("capacity_mva").isNull() | (F.col("capacity_mva") >= 0),
         "Range validation failed: capacity_mva must be >= 0"),
        (F.col("voltage_level_kv").isNull() | (F.col("voltage_level_kv") > 0),
         "Range validation failed: voltage_level_kv must be > 0"),
        (F.col("latitude").isNull() | F.col("latitude").between(-90, 90),
         "Range validation failed: latitude must be between -90 and 90"),
        (F.col("longitude").isNull() | F.col("longitude").between(-180, 180),
         "Range validation failed: longitude must be between -180 and 180"),
    ]
)

In [0]:
# --------------------------------------------------
# 4. TARIFF LOOKUP
# --------------------------------------------------

tariff = read_bronze("bronze_tariff_lookup")

tariff = safe_cast(tariff, "rate_per_kwh", "DOUBLE")
tariff = safe_cast(tariff, "effective_date", "DATE")
tariff = safe_cast(tariff, "expiry_date", "DATE")
tariff = safe_cast(tariff, "peak_multiplier", "DOUBLE")
tariff = standardize_flag(tariff, "active_flag")

# A tariff can have multiple versions with different effective dates
tariff = deduplicate(tariff, ["tariff_code", "effective_date"])

silver_tariff = publish(
    tariff,
    "silver_tariff_lookup",
    ["tariff_code", "effective_date"],
    [
        (F.col("rate_per_kwh").isNull() | (F.col("rate_per_kwh") >= 0),
         "Range validation failed: rate_per_kwh must be >= 0"),
        (F.col("peak_multiplier").isNull() | (F.col("peak_multiplier") > 0),
         "Range validation failed: peak_multiplier must be > 0"),
        (F.col("expiry_date").isNull() |
         (F.col("expiry_date") >= F.col("effective_date")),
         "Date validation failed: expiry_date before effective_date"),
    ]
)

In [0]:
# --------------------------------------------------
# 5. METER MASTER
# --------------------------------------------------

meter = read_bronze("bronze_meter_master")

meter = safe_cast(meter, "installation_date", "DATE")
meter = safe_cast(meter, "rated_voltage", "DOUBLE")

meter = deduplicate(meter, ["meter_id"])

silver_meter = publish(
    meter,
    "silver_meter_master",
    ["meter_id", "substation_id"],
    [
        (F.col("rated_voltage").isNull() | (F.col("rated_voltage") > 0),
         "Range validation failed: rated_voltage must be > 0"),
    ]
)

In [0]:
# --------------------------------------------------
# 6. SMART METER READINGS
# --------------------------------------------------

readings = read_bronze("bronze_smart_meter_readings")

readings = deduplicate(
    readings,
    ["reading_id"],
    "received_ts"
)

silver_readings = publish(
    readings,
    "silver_smart_meter_readings",
    [
        "reading_id",
        "meter_id",
        "substation_id",
        "interval_start_ts",
        "interval_end_ts"
    ],
    [
        (F.col("interval_end_ts") >= F.col("interval_start_ts"),
         "Date validation failed: interval_end_ts before interval_start_ts"),
        (F.col("energy_kwh").isNull() | (F.col("energy_kwh") >= 0),
         "Range validation failed: energy_kwh must be >= 0"),
        (F.col("voltage_v").isNull() | (F.col("voltage_v") >= 0),
         "Range validation failed: voltage_v must be >= 0"),
        (F.col("current_a").isNull() | (F.col("current_a") >= 0),
         "Range validation failed: current_a must be >= 0"),
        (F.col("power_factor").isNull() |
         F.col("power_factor").between(0, 1),
         "Range validation failed: power_factor must be between 0 and 1"),
    ]
)

In [0]:
# --------------------------------------------------
# 7. GRID OUTAGE EVENTS
# --------------------------------------------------

outages = read_bronze("bronze_grid_outage_events")

outages = deduplicate(
    outages,
    ["outage_id"],
    "reported_ts"
)

silver_outages = publish(
    outages,
    "silver_grid_outage_events",
    [
        "outage_id",
        "substation_id",
        "outage_start_ts"
    ],
    [
        (F.col("outage_end_ts").isNull() |
         (F.col("outage_end_ts") >= F.col("outage_start_ts")),
         "Date validation failed: outage_end_ts before outage_start_ts"),
        (F.col("affected_meters").isNull() |
         (F.col("affected_meters") >= 0),
         "Range validation failed: affected_meters must be >= 0"),
    ]
)

In [0]:
# --------------------------------------------------
# 8. METER RECOVERY EVENTS
# --------------------------------------------------

recovery = read_bronze("bronze_meter_recovery_events")

recovery = deduplicate(
    recovery,
    ["recovery_event_id"],
    "received_ts"
)

silver_recovery = publish(
    recovery,
    "silver_meter_recovery_events",
    [
        "recovery_event_id",
        "meter_id",
        "outage_id",
        "event_ts"
    ],
    [
        (F.col("signal_strength").isNull() |
         (F.col("signal_strength") >= 0),
         "Range validation failed: signal_strength must be >= 0"),
    ]
)

In [0]:
# --------------------------------------------------
# 9. REFERENTIAL INTEGRITY CHECKS
# --------------------------------------------------

print("\nREFERENTIAL INTEGRITY CHECKS")

meter_keys = silver_meter.select("meter_id").distinct()
substation_keys = silver_substation.select("substation_id").distinct()
tariff_keys = silver_tariff.select("tariff_code").distinct()
outage_keys = silver_outages.select("outage_id").distinct()

# Meter must refer to an existing substation and tariff
print(
    "Meters with invalid substation:",
    silver_meter.join(
        substation_keys, "substation_id", "left_anti"
    ).count()
)

meters_invalid_tariff = silver_meter.join(
    tariff_keys, "tariff_code", "left_anti"
)
print(
    "Meters with invalid tariff:",
    meters_invalid_tariff.count()
)
display(meters_invalid_tariff)

# Readings must refer to existing meters and substations
print(
    "Readings with invalid meter:",
    silver_readings.join(
        meter_keys, "meter_id", "left_anti"
    ).count()
)

print(
    "Readings with invalid substation:",
    silver_readings.join(
        substation_keys, "substation_id", "left_anti"
    ).count()
)

# Outages must refer to existing substations
print(
    "Outages with invalid substation:",
    silver_outages.join(
        substation_keys, "substation_id", "left_anti"
    ).count()
)

# Recovery events must refer to existing meters and outages
print(
    "Recovery events with invalid meter:",
    silver_recovery.join(
        meter_keys, "meter_id", "left_anti"
    ).count()
)

print(
    "Recovery events with invalid outage:",
    silver_recovery.join(
        outage_keys, "outage_id", "left_anti"
    ).count()
)

print("\nSILVER LAYER PROCESSING COMPLETED!")

In [0]:

from pyspark.sql import functions as F

meter = spark.table(
    "capstone9_smartmeter.smart_grid_silver.silver_meter_master"
)

tariff = spark.table(
    "capstone9_smartmeter.smart_grid_silver.silver_tariff_lookup"
)

print("METER TARIFF CODE AND CUSTOMER CLASS")
display(
    meter.groupBy("tariff_code", "customer_class")
         .count()
         .orderBy("tariff_code")
)

print("TARIFF LOOKUP CODE AND CUSTOMER CLASS")
display(
    tariff.groupBy("tariff_code", "customer_class")
          .count()
          .orderBy("tariff_code")
)
